# ALM Cantilever (Continuous Overhang Constraints)

This is an interactive Jupyter Notebook implementation of the Additive Manufacturing topology optimization using the continuous GGP mapping formulation and the Alternating Augmented Lagrangian optimization loops.

In [ ]:
import numpy as np
import dolfin as df
from gemseo import create_scenario
from ggp.gemseo_wrappers.modular_disciplines import GGPVectorizedGeometryDiscipline, GGPPhysicsFastDiscipline
from gemseo.core.discipline.discipline import Discipline
import matplotlib.pyplot as plt
import os

class AlternatingLowerDiscipline(Discipline):
    def __init__(self, geom_disc, dE_drho, E_e_array, volfrac, lb, ub, num_layers, comp_per_layer, layer_height, alpha_deg, C_offset):
        super().__init__(name="AlternatingLowerDiscipline")
        self.geom_disc = geom_disc
        self.dE_drho = dE_drho
        self.E_e_array = E_e_array
        self.volfrac = volfrac
        self.lb = lb
        self.ub = ub
        self.num_layers = num_layers
        self.comp_per_layer = comp_per_layer
        self.layer_height = layer_height
        self.alpha_deg = alpha_deg
        self.C_offset = C_offset
        self.num_elements = geom_disc.num_elements
        
        self.input_grammar.update_from_names(["x_vars"])
        self.output_grammar.update_from_names(["proxy_obj", "vol_cons", "overhang_cons"])
        
    def _run(self, input_data=None):
        xval = input_data["x_vars"].flatten()
        self.geom_disc.execute({"x_vars": xval})
        
        rho_E = self.geom_disc.local_data["rho_E"].flatten()
        rho_V = self.geom_disc.local_data["rho_V"].flatten()
        
        proxy = self.C_offset - np.sum(rho_E * self.dE_drho * self.E_e_array)
        proxy_log = np.log(proxy + 1.0)
        
        v_val = np.sum(rho_V) / self.num_elements
        vol_cons = (v_val - self.volfrac) / self.volfrac * 100.0
        
        x_unscaled = self.lb + xval * (self.ub - self.lb)
        alpha = np.deg2rad(self.alpha_deg)
        np_val = self.comp_per_layer
        nY = self.num_layers
        
        Xk = x_unscaled[0:np_val*nY]
        Lk = x_unscaled[np_val*nY:2*np_val*nY]
        hk = x_unscaled[2*np_val*nY:2*np_val*nY+nY]
        
        num_cons = (nY - 1) * np_val * 2
        cons = np.zeros(num_cons)
        
        row = 0
        for layer in range(nY - 1):
            for k in range(np_val):
                idx = layer * np_val + k
                idx_next = (layer + 1) * np_val + k
                delta = hk[layer] * np.tan(alpha)
                cons[row] = Xk[idx_next] - Xk[idx] + 0.5*(Lk[idx_next] - Lk[idx]) - delta
                row += 1
                cons[row] = -(Xk[idx_next] - Xk[idx]) + 0.5*(Lk[idx_next] - Lk[idx]) - delta
                row += 1
        
        self.local_data["proxy_obj"] = np.array([proxy_log])
        self.local_data["vol_cons"] = np.array([vol_cons])
        self.local_data["overhang_cons"] = cons

    def _compute_jacobian(self, input_data=None, output_data=None):
        xval = self.local_data["x_vars"].flatten()
        jac_geom = self.geom_disc.linearize(input_data={"x_vars": xval}, compute_all_jacobians=True)
        jac_E = jac_geom["rho_E"]["x_vars"]
        jac_V = jac_geom["rho_V"]["x_vars"]
        
        rho_E = self.geom_disc.local_data["rho_E"].flatten()
        proxy = self.C_offset - np.sum(rho_E * self.dE_drho * self.E_e_array)
        dj_drhoE = - (self.dE_drho * self.E_e_array)
        df0dx = (dj_drhoE @ jac_E) / (proxy + 1.0)
        
        dv_drhoV = np.ones(self.num_elements) * (100.0 / (self.volfrac * self.num_elements))
        dv_dx = dv_drhoV @ jac_V
        
        x_unscaled = self.lb + xval * (self.ub - self.lb)
        alpha = np.deg2rad(self.alpha_deg)
        np_val = self.comp_per_layer
        nY = self.num_layers
        num_vars = len(x_unscaled)
        num_cons = (nY - 1) * np_val * 2
        jac_unscaled = np.zeros((num_cons, num_vars))
        
        row = 0
        for layer in range(nY - 1):
            for k in range(np_val):
                idx = layer * np_val + k
                idx_next = (layer + 1) * np_val + k
                
                jac_unscaled[row, idx_next] = 1.0
                jac_unscaled[row, idx] = -1.0
                jac_unscaled[row, np_val*nY + idx_next] = 0.5
                jac_unscaled[row, np_val*nY + idx] = -0.5
                jac_unscaled[row, 2*np_val*nY + layer] = -np.tan(alpha)
                row += 1
                
                jac_unscaled[row, idx_next] = -1.0
                jac_unscaled[row, idx] = 1.0
                jac_unscaled[row, np_val*nY + idx_next] = 0.5
                jac_unscaled[row, np_val*nY + idx] = -0.5
                jac_unscaled[row, 2*np_val*nY + layer] = -np.tan(alpha)
                row += 1
                
        jac_cons = jac_unscaled * (self.ub - self.lb)
        
        self.jac = {
            "proxy_obj": {"x_vars": df0dx.reshape(1, -1)},
            "vol_cons": {"x_vars": dv_dx.reshape(1, -1)},
            "overhang_cons": {"x_vars": jac_cons}
        }


### Problem Setup
Initialize mesh, variables and start optimization.


In [ ]:
L, H = 60.0, 30.0
nelx, nely = 60, 30
volfrac = 0.3
num_layers = 30
comp_per_layer = 4
layer_height = H / num_layers
alpha_deg = 45.0

mesh = df.RectangleMesh(df.Point(0, 0), df.Point(L, H), nelx, nely)
V_u = df.VectorFunctionSpace(mesh, "CG", 1)
def left_boundary(x, on_boundary): return on_boundary and df.near(x[0], 0.0)
bc = [df.DirichletBC(V_u, df.Constant((0.0, 0.0)), left_boundary)]
boundaries = df.MeshFunction("size_t", mesh, mesh.topology().dim() - 1)
boundaries.set_all(0)
class BottomRight(df.SubDomain):
    def inside(self, x, on_boundary): return df.near(x[0], L) and df.near(x[1], 0.0, 1.0)
BottomRight().mark(boundaries, 1)
ds_load = df.Measure("ds", domain=mesh, subdomain_data=boundaries)
L_rhs_vec = df.Constant((0.0, -1.0))

xLB = np.full(num_layers * comp_per_layer, 0.0)
xUB = np.full(num_layers * comp_per_layer, L)
x0 = np.linspace(0.1*L, 0.9*L, comp_per_layer)
x0_rep = np.tile(x0, num_layers)

LLB = np.full(num_layers * comp_per_layer, 0.1)
LUB = np.full(num_layers * comp_per_layer, L)
L0_rep = np.full(num_layers * comp_per_layer, L/comp_per_layer * 0.8)

hLB = np.full(num_layers, layer_height)
hUB = np.full(num_layers, layer_height)
h0_rep = np.full(num_layers, layer_height)

mLB = np.full(comp_per_layer, 1e-3)
mUB = np.full(comp_per_layer, 1.0)
m0 = np.full(comp_per_layer, volfrac)

lb = np.concatenate([xLB, LLB, hLB, mLB])
ub = np.concatenate([xUB, LUB, hUB, mUB])
x_init = np.concatenate([x0_rep, L0_rep, h0_rep, m0])
xval = (x_init - lb) / (ub - lb)

geom_disc = GGPVectorizedGeometryDiscipline(mesh, len(lb), mode='ALM', num_layers=num_layers, comp_per_layer=comp_per_layer, layer_height=layer_height, pp=10.0, r_gp=0.5, lb=lb, ub=ub)
phys_disc = GGPPhysicsFastDiscipline(mesh, V_u, bc, ds_load, L_rhs_vec, Emin=1e-6)

print("Evaluating initial design...")
geom_disc.execute({"x_vars": xval})
rho_E = geom_disc.local_data["rho_E"].flatten()
phys_disc.execute({"rho_E": rho_E})
U = phys_disc.local_data["U"]

C_offset = np.sum(rho_E * phys_disc.E_e_array) * 1.5

opt_disc = AlternatingLowerDiscipline(geom_disc, phys_disc.dE_drho_array, phys_disc.E_e_array, volfrac, lb, ub, num_layers, comp_per_layer, layer_height, alpha_deg, C_offset)

import gemseo
design_space = gemseo.algos.design_space.DesignSpace()
design_space.add_variable("x_vars", size=len(lb), lower_bound=0.0, upper_bound=1.0, value=xval)

scenario = create_scenario(disciplines=[opt_disc], objective_name="proxy_obj", design_space=design_space, formulation_name="DisciplinaryOpt")
scenario.add_constraint("vol_cons", "ineq", positive=False, value=0.0)
scenario.add_constraint("overhang_cons", "ineq", positive=False, value=0.0)

print("Starting optimization...")
scenario.execute(algo_name="Augmented_Lagrangian_order_1", max_iter=2, algo_options={"algo_options": {"max_iter": 3, "ftol_rel": 1e-4}})

opt_x = scenario.optimization_result.x_opt
geom_disc.execute({"x_vars": opt_x})
rho_opt = geom_disc.local_data["rho_E"]
print("Optimization Finished.")


### Optimized Result

![Optimized Design](_static/alm_cantilever_optimized.png)